<a href="https://colab.research.google.com/github/seahun1/seahun1-streamlit-lab02/blob/main/streamlit_lab02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Streamlit과 OpenAI API, PDF 처리 라이브러리를 설치합니다.
!pip install -q streamlit openai pypdf

# Colab 안에서 만든 웹서버를 외부로 연결해줄 localtunnel을 설치합니다.
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.0/346.0 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 55.4 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 3s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

In [2]:
# 외부 IP 주소를 확인합니다. (나중에 웹 주소 접속 시 비밀번호로 사용됩니다.)
import urllib
print("내 Tunnel Password (IP 주소):", urllib.request.urlopen('https://ident.me').read().decode('utf8'))

내 Tunnel Password (IP 주소): 35.201.230.238


In [6]:
%%writefile app.py
import streamlit as st
from openai import OpenAI
import time

st.set_page_config(page_title="부경대 AI 응용 프로그래밍", layout="centered")
st.title("🤖 Colab 기반 Streamlit 실습")

# =========================================================================
# 1. API Key 입력 및 세션 유지 (실습 1번 조건)
# =========================================================================
if "openai_api_key" not in st.session_state:
    st.session_state["openai_api_key"] = ""

api_key_input = st.text_input("OpenAI API Key 입력:", type="password", value=st.session_state["openai_api_key"]) # [cite: 14]
if api_key_input:
    st.session_state["openai_api_key"] = api_key_input

# @st.cache_data 적용 함수 (실습 1번 캐싱 조건)
@st.cache_data(show_spinner="답변을 생성 중입니다...") # [cite: 15]
def get_cached_response(api_key, question):
    if not api_key: return "API Key를 입력해주세요."
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

# =========================================================================
# 사이드바 메뉴 구성 (실습 2, 3, 4번 페이지 역할 대체)
# =========================================================================
menu = st.sidebar.selectbox("메뉴를 선택하세요", ["일반 질문 (실습1)", "도서관 챗봇 (실습2&3)", "ChatPDF (실습4)"])

# -------------------------------------------------------------------------
# 메뉴 1: 일반 질문 처리
# -------------------------------------------------------------------------
if menu == "일반 질문 (실습1)":
    st.subheader("💡 일반 질의응답 (캐싱 적용)")
    user_q = st.text_input("질문을 입력하세요:") # [cite: 10]
    if st.button("질문하기"):
        res = get_cached_response(st.session_state["openai_api_key"], user_q)
        st.info(res)

# -------------------------------------------------------------------------
# 메뉴 2: 도서관 챗봇 (실습 2, 3번 조건 통합)
# -------------------------------------------------------------------------
elif menu == "도서관 챗봇 (실습 2&3)":
    st.subheader("📚 국립부경대학교 도서관 챗봇") # [cite: 21]

    LIBRARY_REGULATIONS = """
    제11조(휴관일) 도서관의 휴관일은 관공서의 공휴일, 개교기념일로 한다.
    제15조(대출 책수 및 기간) 학부생은 5권, 10일 간 대출할 수 있다.
    """ # [cite: 22, 23, 26]

    if "messages" not in st.session_state:
        st.session_state.messages = []

    if st.button("Clear (대화 초기화)"): # [cite: 19]
        st.session_state.messages = []
        st.rerun()

    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]): st.write(msg["content"]) # [cite: 18]

    if prompt := st.chat_input("도서관 규정에 대해 물어보세요! (ex: 학부생 대여 권수?)"): # [cite: 25, 26]
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"): st.write(prompt)

        if not st.session_state["openai_api_key"]:
            st.error("API Key를 입력해주세요.")
        else:
            client = OpenAI(api_key=st.session_state["openai_api_key"])
            messages = [
                {"role": "system", "content": f"너는 부경대 도서관 챗봇이야. 아래 규정집 내용으로만 답변해.\n{LIBRARY_REGULATIONS}"} # [cite: 24]
            ] + st.session_state.messages

            with st.chat_message("assistant"):
                res = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
                ans = res.choices[0].message.content
                st.write(ans)
            st.session_state.messages.append({"role": "assistant", "content": ans})

# -------------------------------------------------------------------------
# 메뉴 3: ChatPDF (실습 4번 조건)
# -------------------------------------------------------------------------
elif menu == "ChatPDF (실습 4)":
    st.subheader("📄 ChatPDF (OpenAI File Search)") # [cite: 28]

    api_key = st.session_state.get("openai_api_key", "")

    if not api_key:
        st.warning("먼저 상단의 OpenAI API Key를 입력해주세요.")
    else:
        client = OpenAI(api_key=api_key)

        # Assistants API용 세션 변수 초기화
        if "asst_id" not in st.session_state: st.session_state.asst_id = None
        if "thread_id" not in st.session_state: st.session_state.thread_id = None
        if "vs_id" not in st.session_state: st.session_state.vs_id = None
        if "pdf_messages" not in st.session_state: st.session_state.pdf_messages = []

        # 1. 파일 업로더 생성 (하나만 입력 받음)
        uploaded_file = st.file_uploader("PDF 파일을 하나만 업로드하세요", type=["pdf"], accept_multiple_files=False) #

        # 2. Clear 버튼 구현 (생성된 OpenAI 서버 자원 삭제 및 세션 비우기)
        if st.button("Clear 및 벡터 스토어 삭제"): # [cite: 31]
            if st.session_state.asst_id:
                try: client.beta.assistants.delete(st.session_state.asst_id)
                except: pass
            if st.session_state.vs_id:
                try: client.beta.vector_stores.delete(st.session_state.vs_id)
                except: pass

            # 초기화
            st.session_state.asst_id = None
            st.session_state.thread_id = None
            st.session_state.vs_id = None
            st.session_state.pdf_messages = []
            st.success("OpenAI 서버의 벡터 스토어와 대화가 삭제되었습니다!")
            st.rerun()

        # 3. PDF 업로드 시 OpenAI Assistants 가동 및 파일 검색 벡터 스토어 빌드
        if uploaded_file and not st.session_state.asst_id:
            with st.spinner("파일 분석 및 벡터화 진행 중... (약 10~20초 소요)"):
                # OpenAI 서버에 파일 업로드
                openai_file = client.files.create(file=uploaded_file, purpose="assistants")

                # 벡터 스토어 생성 후 파일 연결 (OpenAI File Search 핵심 기능)
                vector_store = client.beta.vector_stores.create(name=f"Colab_VS_{uploaded_file.name}") # [cite: 31]
                client.beta.vector_stores.files.create(vector_store_id=vector_store.id, file_id=openai_file.id) #

                # File Search 기능(tool)을 탑재한 Assistant 생성
                assistant = client.beta.assistants.create(
                    name="PDF 실습 Assistant",
                    instructions="제공된 PDF 문서 내용을 바탕으로 정확하게 답변하세요.",
                    model="gpt-4o-mini",
                    tools=[{"type": "file_search"}], #
                    tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}}
                )

                # 대화를 주고받을 방(스레드) 개설
                thread = client.beta.threads.create()

                # 생성된 ID 값들을 세션에 저장하여 유지
                st.session_state.asst_id = assistant.id
                st.session_state.vs_id = vector_store.id
                st.session_state.thread_id = thread.id
                st.success("업로드 및 인덱싱 완료! 대화를 시작할 수 있습니다.")

        # 기존 PDF 대화 내용 화면에 출력
        for msg in st.session_state.pdf_messages:
            with st.chat_message(msg["role"]): st.markdown(msg["content"])

        # 4. 실시간 질의응답 처리
        if st.session_state.asst_id and (prompt := st.chat_input("PDF 파일 내용을 질문하세요:")): # [cite: 33]
            st.session_state.pdf_messages.append({"role": "user", "content": prompt})
            with st.chat_message("user"): st.markdown(prompt)

            # 4-1. 사용자의 질문 메시지를 OpenAI 생성 스레드방에 전송
            client.beta.threads.messages.create(
                thread_id=st.session_state.thread_id,
                role="user",
                content=prompt
            )

            # 4-2. Assistant 실행 요청 (Run)
            run = client.beta.threads.runs.create(
                thread_id=st.session_state.thread_id,
                assistant_id=st.session_state.asst_id
            )

            # 4-3. AI 답변 생성이 끝날 때까지 무한루프로 대기 (Polling)
            with st.chat_message("assistant"):
                with st.spinner("문서에서 답변 검색 중..."):
                    while run.status in ["queued", "in_progress"]:
                        time.sleep(1)
                        run = client.beta.threads.runs.retrieve(
                            thread_id=st.session_state.thread_id,
                            run_id=run.id
                        )

                    if run.status == "completed":
                        # 생성된 최신 답변 내용 수신
                        messages = client.beta.threads.messages.list(thread_id=st.session_state.thread_id)
                        ans_content = messages.data[0].content[0].text.value
                        st.markdown(ans_content)
                        st.session_state.pdf_messages.append({"role": "assistant", "content": ans_content})
                    else:
                        st.error(f"오류가 발생했습니다. 상태 코드: {run.status}")

Overwriting app.py


In [4]:
# Streamlit 앱을 백그라운드(8501번 포트)에서 실행합니다.
!streamlit run app.py &>/dev/null &

In [5]:
# 8501 포트로 열린 Streamlit 주소를 외부 URL로 포워딩합니다.
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴your url is: https://whole-windows-stop.loca.lt
^C
